In [ ]:
import numpy as np
from itertools import product

# --- METRIC FUNCTIONS ---
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2_score(y_true, y_pred):
    # Avoid division by zero if y_true is constant
    if np.var(y_true) == 0:
        return 0.0
    corr_matrix = np.corrcoef(y_true, y_pred)
    corr = corr_matrix[0, 1]
    return corr ** 2


def manual_grid_search_cv(model_class, X, y, param_grid, cv=5):
    """
    Generic function to run Grid Search CV on ANY model class you pass to it.
    """
    # Ensure inputs are numpy arrays
    if hasattr(X, 'values'): X = X.values
    if hasattr(y, 'values'): y = y.values
    X = np.array(X)
    y = np.array(y)
    
    # Generate all parameter combinations
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in product(*values)]
    
    best_score = float('inf')
    best_params = None
    
    print(f"Starting Grid Search with {cv}-Fold CV...")
    
    # Shuffle indices once
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    
    for i, params in enumerate(combinations):
        fold_scores = []
        
        # Calculate fold size
        fold_size = len(X) // cv
        
        for k in range(cv):
            # Define start and end for test set
            start = k * fold_size
            end = (k + 1) * fold_size if k < cv - 1 else len(X)
            
            test_idx = indices[start:end]
            
            # Train indices are everything else
            train_idx = np.concatenate([indices[:start], indices[end:]])
            
            X_train, y_train = X[train_idx], y[train_idx]
            X_test, y_test = X[test_idx], y[test_idx]
            
            # Check if split resulted in empty sets (sanity check)
            if len(X_train) == 0 or len(X_test) == 0:
                print(f"Warning: Empty fold detected at k={k}")
                continue

            # new modell with params
            model = model_class(**params)
            model.fit(X_train, y_train)
            
            preds = model.predict(X_test)
            
            # Check for NaNs in predictions (can happen if tree is weird)
            if np.isnan(preds).any():
                # print(f"Warning: NaNs in predictions for params {params}")
                mse = float('inf') # Penalize this model
            else:
                mse = mean_squared_error(y_test, preds)
                
            fold_scores.append(mse)
            
        # Robust averaging (ignoring infs if any)
        valid_scores = [s for s in fold_scores if s != float('inf') and not np.isnan(s)]
        if len(valid_scores) > 0:
            avg_mse = np.mean(valid_scores)
        else:
            avg_mse = float('inf')
            
        print(f"Params: {params}, Avg MSE: {avg_mse:.4f}")
        
        if avg_mse < best_score:
            best_score = avg_mse
            best_params = params
            
    print(f"Best Params: {best_params}, Best MSE: {best_score:.4f}")
    return best_params

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv("C:\\Users\\lenar\\Documents\\machine learning\\ML_TU_WS25\\Exercise2\\data\\corn_data.csv")
df.head()
df = df.drop(['County','Farmer','Crop','Power source','Main credit source','Crop insurance','Farm records','Main advisory source','Advisory format',
              'Advisory language','Extension provider','Latitude', 'Longitude'],axis=1)
df = df.dropna()

# separate features and target
target_col = 'Yield'
if target_col in df.columns:
    feature_df = df.drop(columns=[target_col])
    y = df[target_col].values

# prepare for onehot encoding
cat_cols = feature_df.select_dtypes(include=['object']).columns
num_cols = feature_df.select_dtypes(exclude=['object']).columns

# One-Hot Encoding
print(f"Encoding categorical columns: {list(cat_cols)}")
encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_cats = encoder.fit_transform(feature_df[cat_cols])

# Combine Numeric + Encoded Features
X_num = feature_df[num_cols].values
X = np.hstack([X_num, encoded_cats])

print(f"X Shape: {X.shape}, y Shape: {y.shape}")

df.info()
df.describe()
df.head()

Encoding categorical columns: ['Education', 'Gender', 'Age bracket', 'Water source']
X Shape: (328, 13), y Shape: (328,)
<class 'pandas.core.frame.DataFrame'>
Index: 328 entries, 0 to 395
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Education          328 non-null    object 
 1   Gender             328 non-null    object 
 2   Age bracket        328 non-null    object 
 3   Household size     328 non-null    int64  
 4   Acreage            328 non-null    float64
 5   Fertilizer amount  328 non-null    int64  
 6   Laborers           328 non-null    int64  
 7   Yield              328 non-null    int64  
 8   Water source       328 non-null    object 
dtypes: float64(1), int64(4), object(4)
memory usage: 25.6+ KB


,Education,Gender,Age bracket,Household size,Acreage,Fertilizer amount,Laborers,Yield,Water source
0,Certificate,Male,36-45,7,2.00,50,2,300,Rain
1,Certificate,Male,36-45,7,0.25,50,2,270,Rain
2,Certificate,Male,36-45,7,3.00,251,2,270,Rain
3,Certificate,Male,36-45,7,1.50,300,3,200,Rain
5,Certificate,Male,46-55,3,0.50,200,2,180,Rain


In [ ]:
from decisiontree import DecisionTreeRegressor
from randomforest import RandomForestRegressor
# Manual split (80/20) for simple scenarios
indices = np.arange(len(X))
np.random.shuffle(indices)
split = int(0.8 * len(X))
train_idx, test_idx = indices[:split], indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# ======================================================
# SCENARIO 1: Random Forest (Simple Split & Grid Search)
# ======================================================
print("\n=== SCENARIO 1: Random Forest ===")

# A. Simple Split
print("\n--- Part A: Simple Train/Test Split (RF) ---")
rf = RandomForestRegressor(n_estimators=10, max_depth=5)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
print(f"RF Test RMSE: {root_mean_squared_error(y_test, rf_preds):.4f}")
print(f"RF Test R2: {r2_score(y_test, rf_preds):.4f}")

# B. Cross Validation
print("\n--- Part B: Cross Validation Grid Search (RF) ---")
rf_param_grid = {
    'n_estimators': [5, 10],
    'max_depth': [3, 5],
    'n_features': [int(np.sqrt(X.shape[1])), int(X.shape[1]/2)]
}
best_rf_params = manual_grid_search_cv(RandomForestRegressor, X, y, rf_param_grid, cv=3)

# Train final RF on ALL data
print(f"\nTraining final RF with: {best_rf_params}")
final_rf = RandomForestRegressor(**best_rf_params)
final_rf.fit(X, y)
final_rf_preds = final_rf.predict(X)
print(f"Final RF RMSE (full data): {root_mean_squared_error(y, final_rf_preds):.4f}")


# ======================================================
# SCENARIO 2: Decision Tree (Simple Split & Grid Search)
# ======================================================
print("\n=== SCENARIO 2: Decision Tree ===")

# A. Simple Split
print("\n--- Part A: Simple Train/Test Split (DT) ---")
# Instantiate a single tree
dt = DecisionTreeRegressor(max_depth=5, min_samples_split=2)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)

print(f"DT Test RMSE: {root_mean_squared_error(y_test, dt_preds):.4f}")
print(f"DT Test R2: {r2_score(y_test, dt_preds):.4f}")

# B. Cross Validation
print("\n--- Part B: Cross Validation Grid Search (DT) ---")
dt_param_grid = {
    'max_depth': [3, 5, 10],
    'min_samples_split': [2, 5, 10]
}

# Use generic CV tool, passing the DecisionTreeRegressor class
best_dt_params = manual_grid_search_cv(DecisionTreeRegressor, X, y, dt_param_grid, cv=3)

# Train final DT on ALL data
print(f"\nTraining final DT with: {best_dt_params}")
final_dt = DecisionTreeRegressor(**best_dt_params)
final_dt.fit(X, y)
final_dt_preds = final_dt.predict(X)
print(f"Final DT RMSE (full data): {root_mean_squared_error(y, final_dt_preds):.4f}")


--- Part A: Simple Train/Test Split (DT) ---
DT Test RMSE: 81.4253
DT Test R2: 0.6718

--- Part B: Cross Validation Grid Search (DT) ---
Starting Grid Search with 3-Fold CV...
Params: {'max_depth': 3, 'min_samples_split': 2}, Avg MSE: 7518.7286
Params: {'max_depth': 3, 'min_samples_split': 5}, Avg MSE: 7531.1473
Params: {'max_depth': 3, 'min_samples_split': 10}, Avg MSE: 7522.9316
Params: {'max_depth': 5, 'min_samples_split': 2}, Avg MSE: 3535.9740
Params: {'max_depth': 5, 'min_samples_split': 5}, Avg MSE: 3571.5099
Params: {'max_depth': 5, 'min_samples_split': 10}, Avg MSE: 2999.4691
Params: {'max_depth': 10, 'min_samples_split': 2}, Avg MSE: 4531.5479
Params: {'max_depth': 10, 'min_samples_split': 5}, Avg MSE: 4215.4939
Params: {'max_depth': 10, 'min_samples_split': 10}, Avg MSE: 3753.7853
Best Params: {'max_depth': 5, 'min_samples_split': 10}, Best MSE: 2999.4691

Training final DT with: {'max_depth': 5, 'min_samples_split': 10}
Final DT RMSE (full data): 43.7982
